<img src="https://raw.githubusercontent.com/doguilmak/InferenceVision/main/assets/Inference%20Vision%20Cover.png"
      alt="github.com/doguilmak/InferenceVision"/>

In contemporary scientific research and applications, there is an increasing demand for accurate geospatial analysis to address various real-world challenges, ranging from environmental monitoring to urban planning and disaster response. The ability to precisely locate and identify objects within geographic areas plays a pivotal role in such endeavors. In this scientific project, we aim to enhance geospatial analysis by integrating object detection techniques with geographic coordinate calculations.

## **Problem Statement**

Traditional methods of geospatial analysis often rely on manual identification and mapping of objects within geographical regions. However, these methods are time-consuming, labor-intensive, and prone to errors. Moreover, they may lack the scalability required for large-scale analyses. Therefore, there is a need for automated solutions that can accurately detect and locate objects within geographic areas, enabling efficient and scalable geospatial analysis.

## **Project Objective**

This project aims to address the identified challenges by developing an automated system that combines object detection with geographic coordinate calculation. The main objectives are:

1. **Object Detection**
   Use advanced object detection algorithms, such as YOLO (You Only Look Once), to automatically identify and locate objects in satellite or aerial imagery.

2. **Geographic Coordinate Calculation**
   Develop methods to calculate the geographic coordinates (latitude and longitude) of detected objects within a defined geographic area. This includes converting the normalized center coordinates of detected objects into accurate real-world geographic coordinates.

3. **Integration and Visualization**
   Combine object detection results with the calculated geographic coordinates to create a complete geospatial dataset. Export the results in formats such as CSV or GeoJSON to support further analysis and visualization on mapping platforms.

## **Methodology**

This section describes the methodology used in the InferenceVision framework to derive geographic coordinates from object detections in Very High Resolution (VHR) satellite imagery.

#### **Step 1: Transformation to WGS 84 (EPSG:4326)**

The image boundary is transformed from its source Coordinate Reference System (CRS) to WGS 84 (EPSG:4326), ensuring that all coordinates are expressed in geographic latitude and longitude. The transformation produces the coordinates of the image corners. The top-left (TL) and bottom-right (BR) corners are used as reference points:

<br>

$$
(lon_{TL}, lat_{TL}),
(lon_{BR}, lat_{BR})
=
\text{transform}
\left(
bounds_{dataset},
CRS_{dataset}
\rightarrow
EPSG:4326
\right)
$$

<br>

where:

- $lon_{TL}, lat_{TL}$ = longitude and latitude of the top-left corner
- $lon_{BR}, lat_{BR}$ = longitude and latitude of the bottom-right corner

Coordinates are stored with a default precision of nine decimal places.

#### **Step 2: Computation of Normalized Object Centers**

Starting from version 1.3, YOLO directly returns bounding boxes in center format:

<br>

$$
(x_c, y_c, w, h)
=
\texttt{box.xywh}
$$

<br>

where:

- $x_c$ = x-coordinate of the bounding box center
- $y_c$ = y-coordinate of the bounding box center
- $w$ = bounding box width
- $h$ = bounding box height

The center coordinates are normalized using the image dimensions:

<br>

$$
N_x = \frac{x_c}{W}
$$

$$
N_y = \frac{y_c}{H}
$$

<br>

where:

- $N_x$ = normalized horizontal coordinate
- $N_y$ = normalized vertical coordinate
- $W$ = image width (pixels)
- $H$ = image height (pixels)

The normalized coordinates are mapped as:

<br>

$$
N_x \rightarrow \text{Longitude}
$$

$$
N_y \rightarrow \text{Latitude}
$$

<br>

**Version 1.3 update:** The `calculate_bbox_center()` method has been removed because object centers are provided directly by the detector.

#### **Step 3: Geographic Coordinate Interpolation**

The geographic coordinates of each detected object are obtained by linearly interpolating between the transformed image corners.

<br>

Longitude:

$$
lon
=
lon_{TL}
+
(lon_{BR}-lon_{TL})\,N_x
$$

Latitude:

$$
lat
=
lat_{TL}
+
(lat_{BR}-lat_{TL})\,N_y
$$

<br>

where:

- $lon$ = longitude of the detected object
- $lat$ = latitude of the detected object
- $N_x, N_y$ = normalized center coordinates
- $lon_{TL}, lat_{TL}$ = top-left corner coordinates
- $lon_{BR}, lat_{BR}$ = bottom-right corner coordinates

The interpolation assumes a linear relationship between image coordinates and geographic coordinates over the image extent.

## **Scientific Significance**

The proposed project has several scientific implications and contributions.

**Automation and Efficiency**

By automating object detection and geographic coordinate calculation, the system significantly reduces the time and effort required for geospatial analysis, thereby enhancing efficiency and scalability.

**Accuracy and Precision**

Through the integration of advanced algorithms, the system ensures high accuracy and precision in object detection and geographic coordinate calculation, leading to reliable and trustworthy results.

**Versatility and Adaptability**

The developed system is versatile and adaptable to various applications, including environmental monitoring, urban planning, agriculture, and disaster response.

## **1. HOW TO USE**

If you are using Google Colab, ensure that the runtime is configured to use a GPU and Python 3.

Navigate to:

`Runtime → Change runtime type → GPU → Save`

#### **1.1. Clone and Install**

In [1]:
get_ipython().system('git clone https://github.com/doguilmak/InferenceVision.git')
get_ipython().run_line_magic('cd', 'InferenceVision')

Cloning into 'InferenceVision'...
remote: Enumerating objects: 792, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 792 (delta 252), reused 119 (delta 119), pack-reused 467 (from 2)
Receiving objects: 100% (792/792), 67.78 MiB | 16.51 MiB/s, done.
Resolving deltas: 100% (448/448), done.
/content/InferenceVision


In [ ]:
get_ipython().system('pip install -r requirements.txt -q')

#### **1.2. Initialization**

Initialize an instance of the InferenceVision class by providing:

- `tif_path`: path to the input TIFF image file  
- `model_path`: path to the YOLO model weights (.pt file)  
- `coord_precision`: (optional) number of decimal places for coordinates (default = 9)

NOTE: The input image must have a defined CRS to ensure accurate geographic coordinate calculation.

In [3]:
from inference_vision import InferenceVision

print(f"InferenceVision version: {InferenceVision.VERSION}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
InferenceVision version: 1.3


In [4]:
help(InferenceVision)

Help on class InferenceVision in module inference_vision:

class InferenceVision(builtins.object)
 |  InferenceVision(tif_path, model_path, coord_precision=9)
 |
 |  Methods defined here:
 |
 |  __init__(self, tif_path, model_path, coord_precision=9)
 |      Initialize an InferenceVision instance with configurable precision.
 |
 |      Parameters
 |      ----------
 |      tif_path : str
 |          The file path to the TIFF image to be processed.
 |      model_path : str
 |          The file path to the YOLO model weights.
 |      coord_precision : int, optional
 |          The number of decimal places to use for geographic coordinates.
 |          Default is 9.
 |
 |      Raises
 |      ------
 |      ValueError
 |          If the TIFF file cannot be opened.
 |
 |      Example
 |      -------
 |      >>> iv = InferenceVision("image.tif", "model.pt", coord_precision=6)
 |
 |  normalize_center_points(self, center_points)
 |      Normalize the center points based on image dimensions.
 |

In [6]:
iv = InferenceVision(
    tif_path="/content/image.tif",    # Path to your GeoTIFF image
    model_path="/content/model.pt",   # Path to your YOLO model weights
    # coord_precision=6               # Decimal places for coordinates (default 9)
)

#### **1.3. Process Image**

Invoke `process_image()` to run YOLO inference and compute geographic coordinates for each detected object. GPU acceleration improves performance for this step.

**Output Modes**

- `output_format="csv"`  
  Saves results to a CSV file.

- `output_format="geojson"`  
  Saves results as a GeoJSON FeatureCollection (compatible with QGIS, Leaflet, and Google Earth).

- `output_format=None`  
  Prints results to the console (default behavior).

**Notes**

The internal state is reset automatically after each call, allowing the same `iv` instance to be safely reused across multiple images.

##### **CSV Output**

In [7]:
csv_filename = "results.csv"
iv.process_image(output_format="csv", csv_filename=csv_filename)


image 1/1 /content/image.tif: 640x640 5 collapseds, 10 non_collapseds, 62.5ms
Speed: 7.7ms preprocess, 62.5ms inference, 44.7ms postprocess per image at shape (1, 3, 640, 640)

CSV saved as results.csv


In [10]:
import pandas as pd

df = pd.read_csv("results.csv")
df.head()

,Image,Point,Latitude,Longitude,Object Type,Coordinates,Confidence Score,Bounding Box Center,Normalized Bounding Box Center
0,/content/image.tif,0,36.208325,36.153737,collapsed,"[267.3173828125, 503.9970703125, 353.755004882...",0.813706,"[310.53619384765625, 544.2897338867188]","[0.4852128028869629, 0.8504527091979981]"
1,/content/image.tif,1,36.209460,36.152872,non_collapsed,"[24.629562377929688, 88.21810913085938, 101.01...",0.787903,"[62.81989288330078, 118.81305694580078]","[0.09815608263015747, 0.18564540147781372]"
2,/content/image.tif,2,36.209659,36.153166,non_collapsed,"[106.32655334472656, 11.338043212890625, 187.7...",0.780055,"[147.0477294921875, 43.972599029541016]","[0.22976207733154297, 0.06870718598365784]"
3,/content/image.tif,3,36.208671,36.153118,non_collapsed,"[93.32782745361328, 380.359375, 173.0984497070...",0.776515,"[133.213134765625, 414.45849609375]","[0.20814552307128906, 0.6475914001464844]"
4,/content/image.tif,4,36.208312,36.153347,collapsed,"[154.2960205078125, 507.2677307128906, 243.231...",0.772999,"[198.76397705078125, 549.1068115234375]","[0.3105687141418457, 0.8579793930053711]"


##### **GeoJSON Output**

In [8]:
iv.process_image(output_format="geojson", geojson_filename="results.geojson")


image 1/1 /content/image.tif: 640x640 5 collapseds, 10 non_collapseds, 62.6ms
Speed: 2.6ms preprocess, 62.6ms inference, 2.7ms postprocess per image at shape (1, 3, 640, 640)

GeoJSON saved as results.geojson


In [18]:
import json

with open("results.geojson") as f:
    gj = json.load(f)

print(f"Total features: {len(gj['features'])}")
print()

Total features: 15



##### **Console Output**

In [9]:
iv.process_image()


image 1/1 /content/image.tif: 640x640 5 collapseds, 10 non_collapseds, 62.5ms
Speed: 1.8ms preprocess, 62.5ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 640)

Point 0 --------------------
Latitude:  36.208324990 | Longitude: 36.153736908
Object Type: collapsed
Coordinates (Bounding Box): [267.3173828125, 503.9970703125, 353.7550048828125, 584.5823974609375]
Confidence Score: 0.8137
Bounding Box Center (X, Y): (310.54, 544.29)
Normalized Bounding Box Center (X, Y): (0.485213, 0.850453)

Point 1 --------------------
Latitude:  36.209459839 | Longitude: 36.152872278
Object Type: non_collapsed
Coordinates (Bounding Box): [24.629562377929688, 88.21810913085938, 101.01022338867188, 149.4080047607422]
Confidence Score: 0.7879
Bounding Box Center (X, Y): (62.82, 118.81)
Normalized Bounding Box Center (X, Y): (0.098156, 0.185645)

Point 2 --------------------
Latitude:  36.209659456 | Longitude: 36.153166267
Object Type: non_collapsed
Coordinates (Bounding Box): [106.326553344

## **Conclusion**

This notebook demonstrates the end-to-end workflow of the InferenceVision v1.3 framework, including loading a GeoTIFF image, running YOLO inference, computing precise WGS 84 coordinates for each detection, and exporting the results in either CSV or GeoJSON format.

Geographic coordinates (latitude and longitude) are essential for accurately locating detected objects on the Earth's surface. This framework transforms normalized image-space coordinates into real-world geographic coordinates, enabling spatial analysis and mapping of detection results.

## **Debugging**

If errors or unexpected behavior occur during image processing, verify the input data, model configuration, and function usage.

**Common Issues**

- **CRS not set on input TIFF**  
  Geographic coordinate computation may be incorrect or may fail if the input raster does not include a defined Coordinate Reference System (CRS).

- **PROJ / rasterio version mismatch (e.g., `DATABASE.LAYOUT.VERSION` error)**  
  This is typically caused by an incompatible geospatial dependency setup.  
  Updating `rasterio` often resolves the issue:

- **No objects detected**  
This may indicate issues with:
  - model weights (incorrect or untrained for the target domain)
  - image resolution mismatch relative to training data

- **Missing CSV filename error** (`ValueError: csv_filename must be provided`)  
When using `output_format="csv"`, a valid `csv_filename` must always be provided.

<br>

---

*Library will be available as a package on PyPI (Python Package Index).*